### Stellar Stream Tutorial  | Probe Combination for Dark Matter Physics in the Era of Large Surveys

This tutorial will provide an overview of the following tasks:
1. Generating an unperturbed stream.
2. Applying a single subhalo impact.
4. Generating many CDM stream realizations with direct integration (somewhat slow).
5. Generating many CDM stream realizations with linear perturbation theory (cost: upfront compute)
6. Experimental: Optimizing the subhalo impact to roughly match observations. 



This tutorial uses [`streamsculptor`](https://github.com/jnibauer/streamsculptor) (differentiable stream modeling in JAX).

This notebook can be run on a laptop (cpu) though heavy inference tasks are best suited for a single GPU. 


You can also run this notebook on Google's servers (with a GPU!) through [Kaggle](https://www.kaggle.com/code/jacobnibauer/streams-tutorial).
- To enable GPU usage, you will need to create / verify your account (recommended). 

-------


In [ ]:
from astropy import units as u
from streamsculptor.main import usys
import jax.numpy as jnp
from astropy.coordinates import SkyCoord, Galactocentric

import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)

from streamsculptor import potential
from streamsculptor import JaxCoords as jc
from streamsculptor import usys
from streamsculptor import gen_stream_Chen25

import diffrax
import matplotlib as mpl
import streamsculptor as ssc

from astropy.table import Table 
import numpy as np
import interpax
import equinox as eqx
import tqdm

First we define a GD-1-centric frame. These transforms are written in JAX so they can be differentiated.

In [ ]:
@jax.jit
def icrs_to_gd1(ra_rad, dec_rad):
    """
    Differentiable ra, dec --> gd1 phi1, phi2 (Koposov+2010 rotation matrix).
    ra_rad, dec_rad in radians.
    """
    R = jnp.array(
        [
            [-0.4776303088, -0.1738432154, 0.8611897727],
            [0.510844589, -0.8524449229, 0.111245042],
            [0.7147776536, 0.4930681392, 0.4959603976],
        ]
    )

    icrs_vec = jnp.vstack([jnp.cos(ra_rad)*jnp.cos(dec_rad),
                           jnp.sin(ra_rad)*jnp.cos(dec_rad),
                           jnp.sin(dec_rad)]).T

    stream_frame_vec = jnp.einsum('ij,kj->ki', R, icrs_vec)

    phi1 = jnp.arctan2(stream_frame_vec[:,1], stream_frame_vec[:,0])*180/jnp.pi
    phi2 = jnp.arcsin(stream_frame_vec[:,2])*180/jnp.pi

    return phi1, phi2

@jax.jit
def get_phi12_from_stream(stream):
    """Differentiable helper: simulated stream --> phi1, phi2."""
    ra_s, dec_s, dist_ = jax.vmap(jc.simcart_to_icrs)(stream[:,:3])
    phi1_model, phi2_model = icrs_to_gd1(ra_s*jnp.pi/180, dec_s*jnp.pi/180)
    return phi1_model, phi2_model

### Dataset: *Gaia* DR3 GD-1 members
We use the membership catalog of Starkman & Nibauer + 2023

In [ ]:
dat = "gd1_members_slim.ecsv"

tab = Table.read(dat, format="ascii.ecsv")
gd1 = tab[tab["allstream_mle"] >= 0.75]              # keep >75% membership
ra_rad  = jnp.deg2rad(np.asarray(gd1["ra"]))
dec_rad = jnp.deg2rad(np.asarray(gd1["dec"]))
phi1_data, phi2_data = icrs_to_gd1(ra_rad, dec_rad)
print(len(gd1), "GD-1 members")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(phi1_data, phi2_data, s=5, zorder=-5,color="k", label=r"Gaia DR3 members (from Starkman \& Nibauer + 2023)")
ax.set_xlim(-90, 15); ax.set_ylim(-6, 6)
ax.set_xlabel(r"$\phi_1$ [deg]"); ax.set_ylabel(r"$\phi_2$ [deg]")
ax.legend(loc="upper left", markerscale=3); ax.set_aspect("equal")
plt.show()

## Generate a smooth GD-1 model

Estimated progenitor location today `[kpc, kpc, kpc, kpc/Myr, kpc/Myr, kpc/Myr]`
(from `data/GD1_prog/GD1_progenitor.npy` in the streamsculptor repo).

In [ ]:
prog_wtoday = jnp.array([-11.88431447,   1.61212652,   7.14988315,  -0.09253346, -0.22701121,  -0.10732978])

Set up the potential, stream age, dissolution time, and stripping times. Then generate an
**unperturbed Chen+25 stream**.

Note the switch from the Fardal notebook: instead of `pot.gen_stream_vmapped` (Fardal release),
we call `ssc.gen_stream_Chen25(..., method="vmap")`. The Chen+25 model takes a `key` (PRNGKey)
rather than a `seed_num`.

In [ ]:
## Use Gala's MW potential
pot = potential.GalaMilkyWayPotential(units=usys)
## Age of the stream [Myr]
t_age = 4_000.0
## Past time at which it dissolved [Myr]
t_dissolve = -300.0
## Past IC (unperturbed backward integration in the base potential)
IC = pot.integrate_orbit(w0=prog_wtoday, t0=0.0, t1=-t_age, ts=jnp.array([-t_age])).ys[0]
IC

In [ ]:
## Stripping times. ts[-1] is always the final integration time for gen_stream simulations
N_tail = 1_500
ts = jnp.hstack([jnp.linspace(-t_age, t_dissolve, N_tail), jnp.array([0.0])])

## Generate the unperturbed Chen+25 stream [l: lead, t: trail]
l, t = ssc.gen_stream_Chen25(pot_base=pot, ts=ts, prog_w0=IC, Msat=1e4,
                             key=jax.random.PRNGKey(532), method="vmap",
                             solver=diffrax.Dopri8(), atol=1e-7, rtol=1e-7, dtmin=0.1)
stream = jnp.vstack([l, t])

In [ ]:
phi1_model, phi2_model = get_phi12_from_stream(stream)


fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(phi1_model, phi2_model, s=1.2, color="tab:blue", alpha=0.4, label="Model", rasterized=True)
ax.scatter(phi1_data, phi2_data, s=5, zorder=-5,color="k", label="Gaia DR3 members")
ax.set_xlim(-90, 15); ax.set_ylim(-6, 6)
ax.set_xlabel(r"$\phi_1$ [deg]"); ax.set_ylabel(r"$\phi_2$ [deg]")
ax.legend(loc="upper left", markerscale=3); ax.set_aspect("equal")
plt.show()



In [ ]:
order = jnp.argsort(phi1_model)
splines = [interpax.Interpolator1D(x=phi1_model[order], f=stream[order, i], method="cubic2")
           for i in range(6)]

@eqx.filter_jit
def eval_mean(phi1_0):
    "differentiable mean phase-space coordinate of the smooth stream at phi1_0"
    return jnp.array([splines[i](phi1_0) for i in range(6)])

sigma_sub = (180 * u.km/u.s).to(u.kpc/u.Myr).value    # subhalo velocity scale

In [ ]:
def get_impact_params(p):
    "impact params dict -> SubhaloLinePotential (single subhalo on a line)"
    W0 = pot.integrate_orbit(w0=eval_mean(p["phi1_0"]), t0=0.0, t1=p["t_impact"],
                             ts=p["t_impact"][None], solver=diffrax.Dopri8(),
                             atol=1e-7, rtol=1e-7, dtmin=0.1).ys[0]
    T = W0[3:] / jnp.linalg.norm(W0[3:])
    B = jnp.cross(W0[:3], W0[3:]); B = B / jnp.linalg.norm(B)
    Nn = jnp.cross(B, T)
    b_hat   = Nn*jnp.cos(p["psi"]) + B*jnp.sin(p["psi"])   # impact offset direction
    perp    = -Nn*jnp.sin(p["psi"]) + B*jnp.cos(p["psi"])  # in-plane perpendicular
    X = W0[:3] + p["b"] * b_hat
    w = p["wspeed"] * (p["wpar_frac"]*T + jnp.sqrt(jnp.maximum(1 - p["wpar_frac"]**2, 0.))*perp)
    V = W0[3:] + w
    return X, V


def build_subhalo(p):
    """
    Subhalo params dict -> SubhaloLinePotential (single subhalo on a line)
    """
    X, V = get_impact_params(p)
    return potential.SubhaloLinePotential(
        m=jnp.array([10**p["log10m"]]), a=jnp.array([p["a"]]),
        subhalo_x0=X[None], subhalo_v=V[None], subhalo_t0=p["t_impact"][None],
        t_window=jnp.array([250.0]), units=usys)

KEY   = jax.random.PRNGKey(0)
SOLVE = dict(solver=diffrax.Dopri5(), rtol=1e-6, atol=1e-6, dtmin=0.5)

@eqx.filter_jit
def perturbed_stream(p, pot_SH):
    "Backward-integrate the progenitor WITH the impact (so it always lands at today's observed"
    "position), then forward-generate the stream (Chen+25, vmap). This anchors the present-day"
    "progenitor: the impact perturbs the stream, not where the progenitor ends up."
    #SH = build_subhalo(p)
    pot_tot = potential.Potential_Combine(potential_list=[pot, pot_SH], units=usys)
    IC_pert = pot_tot.integrate_orbit(w0=prog_wtoday, t0=0.0, t1=-t_age, ts=jnp.array([-t_age]),
                                      solver=diffrax.Dopri8(), rtol=1e-7, atol=1e-7, dtmin=0.1).ys[0]
    l, t = gen_stream_Chen25(pot, ts, IC_pert, Msat=1e4, key=KEY, method="vmap",
                             pot_pert=pot_SH, **SOLVE)
    return jnp.vstack([l, t])

In [ ]:
p_eye = dict(phi1_0=jnp.array(-50.0), t_impact=jnp.array(-180.0), #-200
             b=jnp.array(0.2), psi=jnp.array(jnp.deg2rad(210.)),
             wspeed=jnp.array(.4*sigma_sub), wpar_frac=jnp.array(.7), #.4, .7
             log10m=jnp.array(7.2), a=jnp.array(.1))

sub_pot = build_subhalo(p_eye)
stream_eye = perturbed_stream(p_eye, sub_pot)
p1_eye, p2_eye = get_phi12_from_stream(stream_eye)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(phi1_data, phi2_data, s=5, color="k", label="data")
ax.scatter(p1_eye, p2_eye, s=1.2, color="tab:red", alpha=0.4, label="by-eye impact", rasterized=True)
ax.set_xlim(-65, -10); ax.set_ylim(-4, 4)
ax.set_xlabel(r"$\phi_1$ [deg]"); ax.set_ylabel(r"$\phi_2$ [deg]"); ax.legend(markerscale=4)
plt.show()


We can also utilize a tidally evolved truncated NFW profile, which is more common in (e.g.) the strong lensing literature. 
Using  `TNFWSubhaloLinePotential.from_infall`, we set the infall mass, concentration, redshift, and present-day bound fraction. 
We use the tidal track from Du+2024 and Planck 2018 cosmological parameters under the hood.

$$\rho_{\rm tNFW}\left(r;\rho_s, r_s, f_{\rm bound}\right) = \rho_{\rm NFW}\left(r;\rho_s, r_s\right) \ T\left(f_{\rm bound}\right)$$
with
$$
T(f_{\rm bound})
=
\frac{f_t(f_{\rm bound})}
{1+\left[r/r_t(f_{\rm bound})\right]^2}.
$$

In [ ]:
from streamsculptor.tnfw_analytic import TNFWSubhaloLinePotential

X,V = get_impact_params(p_eye)

pot_line_tnfw = TNFWSubhaloLinePotential.from_infall(
    m_infall=jnp.array([2e7]), c_infall=jnp.array([120.0]), z_infall=jnp.array([10.0]), f_bound=jnp.array([.3]),
    subhalo_x0=X[None], subhalo_v=V[None], subhalo_t0=p_eye['t_impact'][None],
    t_window=250.0,
    )

stream_tnfw = perturbed_stream(p_eye,pot_line_tnfw)

p1_tnfw, p2_tnfw = get_phi12_from_stream(stream_tnfw)


fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(phi1_data, phi2_data, s=5, color="k", label="data")
ax.scatter(p1_tnfw, p2_tnfw, s=1.2, color="tab:red", alpha=1.0, rasterized=True)
ax.set_xlim(-65, -10)
ax.set_ylim(-4, 4)
ax.set_xlabel(r"$\phi_1$ [deg]"); ax.set_ylabel(r"$\phi_2$ [deg]"); ax.legend(markerscale=4)
plt.show()


In [ ]:
# cycle through a few z_infall
z_infall_high = 10.0
z_infall_low = 0.5
z_infall_trial = jnp.logspace(jnp.log10(z_infall_high),jnp.log10(z_infall_low),4)
p1_tnfw_arr = np.zeros((len(p1_tnfw),len(z_infall_trial)))
p2_tnfw_arr = np.zeros((len(p1_tnfw),len(z_infall_trial)))
for i in tqdm.tqdm(range(len(z_infall_trial))):
    z_infall = z_infall_trial[i]
    
    pot_line_tnfw = TNFWSubhaloLinePotential.from_infall(
                m_infall=jnp.array([3e7]), c_infall=jnp.array([120.0]), z_infall=z_infall, f_bound=jnp.array([.8]),
                subhalo_x0=X[None], subhalo_v=V[None], subhalo_t0=p_eye['t_impact'][None],
                t_window=250.0,
               
                )

    stream_tnfw = perturbed_stream(p_eye,pot_line_tnfw)

    p1_tnfw, p2_tnfw = get_phi12_from_stream(stream_tnfw)
    p1_tnfw_arr[:,i] = p1_tnfw
    p2_tnfw_arr[:,i] = p2_tnfw




In [ ]:
z_infall_trial

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))

for i in range(len(z_infall_trial)):
  y_stream = p2_tnfw_arr[:, i] - 5 + i * 5
  sc = ax.scatter(p1_tnfw_arr[:, i], y_stream, s=1.2, alpha=1.0, rasterized=True)

  # Find the rightmost point in the stream to anchor the text
  idx_right = np.nanargmax(p1_tnfw_arr[:, i])
  x_pos = -10.5
  y_pos = y_stream[idx_right] + 4.5

  ax.text(
      x_pos,
      y_pos + 0.9,
      rf"$z_{{\rm infall}} = {z_infall_trial[i]:.2f}$",
      color=sc.get_facecolor()[0],
      fontsize=14,
      ha="right",
      va="bottom",
      fontweight="medium",
  )

ax.set_xlim(-70, -10)
ax.set_ylim(-8, 15)
ax.set_xlabel(r"$\phi_1$ [deg]")
ax.set_ylabel(r"$\phi_2$ [deg]")
plt.tight_layout()
plt.show()

Now introduce $N$ subhalos with the `ImpactGenerator`. It samples impacts by taking the
average phase-space location of particles in $\phi_1$ bands (a "phase-space patch"), integrating
back to an impact time, and randomly sampling impact parameters, angles, and speeds in the frame
of that patch. It needs the unperturbed stream's $\phi_1$ coordinate to parametrize the impacts.

Here's an example with 20 subhalos.

In [ ]:
from streamsculptor.GenerateImpactParams import ImpactGenerator

ImpactGen = ImpactGenerator(pot=pot,
                            tobs=0.0,
                            stream=stream,
                            stream_phi1=phi1_model,
                            phi1_bounds=[-80.,20.],
                            tImpactBounds=[-t_age,0.0],
                            phi1window=.8,
                            NumImpacts=20,
                            bImpact_bounds=[0,.1],
                            stripping_times=jnp.hstack([ts[:-1],ts[:-1]]),
                            prog_today=prog_wtoday,
                            seednum=2332)#22

ImpactDict = ImpactGen.get_subhalo_ImpactParams()
print(ImpactDict.keys())

In [ ]:
ImpactDict['CartesianImpactParams']

In [ ]:
ImpactDict['ImpactFrameParams']

Impact times are sampled from a pdf proportional to the stream's estimated length over time.

In [ ]:
tImp = ImpactDict['ImpactFrameParams']['tImpact']
length_osc = ImpactGen.length_osc
plt.plot(length_osc['ts'], length_osc['length_func']/3, label='Stream length')
plt.hist(tImp, bins=30, label=r'$p(t_{\rm impact})$');
plt.xlabel(r'$t$ [Myr]', fontsize=20)
plt.legend()

Now a function to generate the perturbed stream given subhalo masses and scale radii.
We build the subhalo potential from the impact dictionary, **backwards-integrate the observed
progenitor through the base + subhalo potential**, then
forward-generate the stream with `gen_stream_Chen25(..., method="vmap", pot_pert=...)`.

`method="vmap"` keeps the whole thing jittable and differentiable.

In [ ]:
@jax.jit
def gen_perturbed_stream(m, rs):
    """
    Generate a perturbed stream given 
    an array of subhalo masses and scale radii. 
    Outputs N x 6 array [kpc, kpc/Myr]
    """
    pot_SH = potential.SubhaloLinePotential(m=m,
                                            a=rs,
                                            subhalo_x0=ImpactDict['CartesianImpactParams'][:,:3],
                                            subhalo_v=ImpactDict['CartesianImpactParams'][:,3:],
                                            subhalo_t0=ImpactDict['ImpactFrameParams']['tImpact'],
                                            t_window=jnp.array([200.0]),
                                            units=usys)

    pot_total = potential.Potential_Combine([pot, pot_SH], units=usys)

    # Do NOT fix the prog: perturbed past IC via backward integration in base + subhalos
    IC_pert = pot_total.integrate_orbit(w0=prog_wtoday, t0=0.0, t1=-t_age,
                                        ts=jnp.array([-t_age]), max_steps=5_000).ys[0]

    l, t = ssc.gen_stream_Chen25(pot_base=pot, pot_pert=pot_SH, prog_w0=IC_pert, ts=ts,
                                 Msat=1e4, key=jax.random.PRNGKey(532), method="vmap",
                                 max_steps=5_000, atol=1e-6, rtol=1e-6,
                                 solver=diffrax.Dopri8(), dtmin=.1)
    return jnp.vstack([l, t])

Run it: 20 subhalos, masses $5\times10^6\,M_\odot$, scale radii $0.05$ kpc. 

In [ ]:
m = jnp.ones(20)*5e6
rs = jnp.ones(20)*.1
perturbed_stream = gen_perturbed_stream(m, rs)

In [ ]:
phi1_pert, phi2_pert = get_phi12_from_stream(perturbed_stream[:,:3])

fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)
ax.scatter(phi1_pert, phi2_pert, s=.2, color='grey')

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-5,5)
ax.set_xlim(-80,20)
ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=20)
ax.set_ylabel(r'$\phi_2$ [deg]', fontsize=20)
ax.set_aspect('equal')

# CDM-motivated realizations
We will use the function `gen_perturbed_stream_chen25` (note we also have `gen_perturbed_stream_fardal` and `gen_perturbed_stream`; the former is based on the Fardal+15 method, and the latter can be utilized to generate perturbations to a _N_-body stream)

We draw subhalo masses from a CDM mass function with the `RateCalculator`.
`gen_perturbed_stream_chen25` bundles the subhalo sampling, impact placement, subhalo
potential construction, perturbed-IC backward integration, and forward stream
generation into one call.

In [ ]:
from streamsculptor.subhalostatistics import RateCalculator

In [ ]:
# progenitor orbit + stream length in the observed window
tsave = jnp.linspace(-t_age, 0.0, 500)
prog_orb = pot.integrate_orbit(w0=IC, t0=-t_age, t1=0.0, ts=tsave)
inside = (phi1_model > -80) & (phi1_model < 20.)

length = ssc.compute_stream_length(stream=stream[inside], phi1=phi1_model[inside])
print(f'Stream length: {length:.2f} kpc')

RC = RateCalculator(orbit=prog_orb,
                    t_age=t_age,
                    b_max_fac=5.0,
                    l_obs=length)

In [ ]:
log10M_eval = jnp.linspace(6, 8.5, 1000)

dn_dlog10M_CDM = RC.dn_dlog10M(r=20.0, 
                    log10M=log10M_eval
                    )

dn_dlog10M_WDM = RC.dn_dlog10M(r=20.0, 
                    log10M=log10M_eval,
                    M_hm = 1e6 #, # WDM half-mode mass
                    )

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(6,5)
ax.plot(log10M_eval, dn_dlog10M_CDM, label='CDM', color='k',lw=2)
ax.plot(log10M_eval, dn_dlog10M_WDM, label='WDM', color='r',lw=2)

ax.set_yscale('log')
ax.legend(fontsize=20)

ax.set_xlabel(r'$\log_{10} M$', fontsize=20)
ax.set_ylabel(r'$dn/d\log_{10} M$', fontsize=20)

The subhalo number density per unit mass must be converted to a subhalo encounter rate. That is, how many encounters per mass per time the stream experiences.

The RateCalculator provides this conversion, via the Yoon+2011 and Erkal+2016 formalism:
$$\frac{d\dot{N}_{\rm enc}}{dM} \sim \sigma_{\rm sub} b_{\rm max}\left(M\right) \ell\left(t\right) \frac{dn_{\rm sub}}{dM}$$


In [ ]:
from functools import partial

In [ ]:
# The RC assumes single inputs for log10M and M_hm. 
# To cycle through an array of log10M values, we can use jax.vmap to vectorize the function.
# We will only vectorize over log10M (so None for M_hm).
@partial(jax.vmap, in_axes=(0, None))
def dN_enc_dlog10M(log10M, M_hm):
    return RC.dN_encounter_dlog10M(log10M=log10M, M_hm=M_hm)

dNenc_dlog10M_CDM = dN_enc_dlog10M(log10M_eval, 0.0)
dNenc_dlog10M_WDM = dN_enc_dlog10M(log10M_eval, 1e6)


In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(6,5)
ax.plot(log10M_eval, dNenc_dlog10M_CDM, label='CDM', color='k',lw=2)
ax.plot(log10M_eval, dNenc_dlog10M_WDM, label='WDM', color='r',lw=2)

#ax.set_yscale('log')
ax.legend(fontsize=20)

ax.set_xlabel(r'$\log_{10} M$', fontsize=20)
ax.set_ylabel(r'$dN_{\mathrm enc}/d\log_{10} M$', fontsize=20)

In [ ]:
samps = RC.sample_masses(log10M_min=6.0, log10M_max=9.0,
                         key=jax.random.PRNGKey(3032), array_length=50, M_hm=0.0)
samps

In [ ]:
plt.hist(samps['log10_mass'][samps['log10_mass']>0], bins=10, color='k', alpha=0.7);
plt.xlabel(r'$\log_{10}\left(M/M_\odot\right)$', fontsize=11)

In [ ]:
from functools import partial
from streamsculptor.perturbedstream import gen_perturbed_stream_chen25

A single realization.
- takes `prog_wtoday` - the progenitor is anchored at its observed position today,
- takes a `stream_key` PRNGKey


In [ ]:
out = gen_perturbed_stream_chen25(
                         RateCalculator=RC,
                         pot_base=pot,
                         unperturbed_stream=stream,
                         phi1_unperturbed=phi1_model,
                         stream_key=jax.random.PRNGKey(52132),
                         ts=ts,
                         prog_wtoday=prog_wtoday,
                         Msat=1e4,
                         log10M_min=6.0,
                         log10M_max=9.0,
                         phi1_exclude=[-22,-19],
                         phi1window=0.8,
                         subhalo_key=jax.random.PRNGKey(10),
                         r_s_fac=1.0,
                         mass_fac=1.0,
                         phi1_bounds=[-80.,20.],
                         tImpactBounds=[-t_age,0.0],
                         Max_Num_Impacts=50,
        )

In [ ]:
stream_pert = jnp.vstack([out['leading'], out['trailing']])
phi1_pert, phi2_pert = get_phi12_from_stream(stream_pert[:,:3])

fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)
ax.scatter(phi1_pert, phi2_pert, s=.1, color='grey')
ax.scatter(phi1_model, phi2_model+3, s=.1, color='tab:blue')

ax.text(22, -0.2, 'unpert', color='tab:blue', fontsize=12, va='center')
ax.text(22, -3, 'pert', color='gray', fontsize=12, va='center')

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-5,5)
ax.set_xlim(-80,20)
ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=15)
ax.set_ylabel(r'$\phi_2$ [deg]', fontsize=13)
ax.set_aspect('equal')

Wrap it in a jitted realization function, keyed on the subhalo RNG, and map over many keys.

In [ ]:
@jax.jit
def gen_realization(subhalo_key, r_s_fac=1., mass_fac=1.0):
    out = gen_perturbed_stream_chen25(
                         RateCalculator=RC,
                         pot_base=pot,
                         unperturbed_stream=stream,
                         phi1_unperturbed=phi1_model,
                         stream_key=jax.random.PRNGKey(532),
                         ts=ts,
                         prog_wtoday=prog_wtoday,
                         Msat=1e4,
                         log10M_min=6.0,
                         log10M_max=8.,
                         subhalo_key=subhalo_key,
                         r_s_fac=r_s_fac,
                         mass_fac=mass_fac,
                         phi1_bounds=[-80.,20.],
                         phi1_exclude=[-25,-15],
                         tImpactBounds=[-t_age,0.0],
                         Max_Num_Impacts=50,
                         dtmin=.25
        )
    stream_pert = jnp.vstack([out['leading'], out['trailing']])
    phi1_pert, phi2_pert = get_phi12_from_stream(stream_pert[:,:3])
    return phi1_pert, phi2_pert

In [ ]:
phi1_p, phi2_p = gen_realization(jax.random.PRNGKey(223232), .5, 1.)

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)
ax.scatter(phi1_model, phi2_model+3, s=.02, color='tab:blue')
ax.scatter(phi1_p, phi2_p, s=.02, color='grey')

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-5,5)
ax.set_xlim(-80,20)
ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=20)
ax.set_ylabel(r'$\phi_2$ [deg]', fontsize=20)
ax.set_aspect('equal')

### We can now generate multiple realizations of the perturbed stream by vectorizing the gen_realization function over an array of random keys. This allows us to efficiently compute multiple perturbed streams in parallel.


In [ ]:
mapped_realization_func = jax.vmap(gen_realization, in_axes=(0,None,None))

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(38), 5)
len(keys)

In [ ]:
realizations = mapped_realization_func(keys, .3, 1.0)

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)
for i in range(len(realizations[0])):
    phi1_p, phi2_p = realizations[0][i], realizations[1][i]
    ax.scatter(phi1_p, phi2_p-5+i*3, s=.1)

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-10,10)
ax.set_xlim(-80,20)
ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=20)
ax.set_ylabel(r'$\phi_2$ [deg]', fontsize=20)
ax.set_aspect('equal')

# Building a derivative library with `get_derivs`, then fast realizations

The `gen_perturbed_stream_chen25` calls above re-integrate the stream for every draw of the
subhalo population. This is expensive (though somewhat feasible on a GPU if you're okay waiting)
to repeat thousands of times.

For **population statistics** we instead build a *linear-response library* once, then re-weight
it. `streamsculptor.generate_derivs.get_derivs` computes how the stream responds to a pool of candidate
subhalo impacts: for each it integrates `d(stream)/d(mass)` and `d^2(stream)/d(mass)d(r_s)`.

Each pool slot is a specific subhalo with its own geometry, build mass, and **root radius**
`r_s_root` (the value its structural derivative was evaluated at). With `save=False` it returns
the batches in memory instead of writing `.npy` files. Integrating the response for 100 impacts
takes ~2 minutes on CPU.
~3 minutes on CPu for 200.

In [ ]:
from streamsculptor.generate_derivs import get_derivs

## Build the Chen+25 derivative library: a pool of N_subhalos candidate impacts across the
## mass range, each with d(stream)/d(mass) and d(stream)/d(r_s) integrated once. save=False
## returns the batch dict(s) in memory: pert_out = [stream_base, stream_derivs], plus r_s_root.
N_subhalos = 200#50

batches = get_derivs(
                prog_wtoday=prog_wtoday,
                t_age=t_age,
                t_dissolve=t_dissolve,
                log10_min_mass=6.0,
                log10_max_mass=8.0,
                phi1_bounds=[-70., 20.],
                phi1_exclude=[-25., -15.],        
                stream_seednum=532,
                key=jax.random.PRNGKey(21332), #394
                Msat=1e4,
                bmax_fac = 5.0,
                target_num=N_subhalos,
                phi1_function=lambda s: get_phi12_from_stream(s)[0],
                pot=pot,
                self_grav=False,                       # no progenitor self-gravity, matching the rest of this notebook
                save=False,                            # <-- return in memory instead of writing to disk
                N_batch=N_subhalos,
                N_arm=800,#1_500,#1500,
                phi1window=0.8,
                atol=1e-10,
                rtol=1e-10)

In [ ]:
batches[0]['pert_out'][0].shape

In [ ]:
batches[0]['r_s_root'].shape

In [ ]:

## Chen+25 get_derivs packs the full (lead+trail) stream into pert_out directly:
##   pert_out[0] = stream_base   (N_particle, 6)
##   pert_out[1] = stream_derivs (N_particle, N_subhalo, 12)   cols 0:6 = d/dm, 6:12 = d/dr_s
## Concatenate the per-batch subhalo pools along the slot axis; base is shared across batches.
stream_base   = batches[0]['pert_out'][0]
stream_derivs = batches[0]['pert_out'][1]#jnp.concatenate([b['pert_out'][1] for b in batches], axis=1)
r_s_root      = batches[0]['r_s_root']#jnp.concatenate([b['r_s_root']     for b in batches], axis=0)   # per-slot fiducial radius
N_subhalos    = stream_derivs.shape[1]

print('stream_base:  ', stream_base.shape)
print('stream_derivs:', stream_derivs.shape)
print('r_s_root:     ', r_s_root.shape, '  range %.2f-%.2f kpc' % (float(r_s_root.min()), float(r_s_root.max())))

In [ ]:
@jax.jit
def compute_pert_stream(stream_base, stream_derivs, mass_subhalos, delta_rs):
    """
    Helper function to compute the perturbed stream, once derivatives have already been calculated.
    stream_base: the base stream [N_particle x 6]
    stream_derivs: the derivatives of the stream [N_particle x N_subhalo x 12]
    mass_subhalos: the mass of each subhalo [N_subhalo]
    delta_rs: the radius perturbation for each subhalo [N_subhalo]

    outputs:
    stream_pert: the perturbed stream [N_particle x 6]
    """
    stream_response = stream_derivs[:,:,:6]*mass_subhalos[None,:,None] + stream_derivs[:,:,6:]*mass_subhalos[None,:,None]*delta_rs[None,:,None]
    ## Sum over the perturbations, add em all up
    stream_pert = stream_base + jnp.sum(stream_response,axis=1)
    return stream_pert

In [ ]:
import tqdm


In [ ]:
mass_subhalos = jnp.zeros(N_subhalos)
delta_rs = jnp.ones(N_subhalos)*(-0.)

test_mass = jnp.logspace(6,8.,10)
subhalo_view = 103 #34
pert_stream_list = []
for i in tqdm.tqdm(range(len(test_mass))):
    mass_subhalos = mass_subhalos.at[subhalo_view].set(test_mass[i])
    stream_pert = compute_pert_stream(stream_base, stream_derivs,mass_subhalos,delta_rs)
    pert_stream_list.append(stream_pert)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable


fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)

offset = 8.8

# Normalize log_mass values
norm = mcolors.Normalize(vmin=min(np.log10(test_mass)), vmax=max(np.log10(test_mass)))

for i in range(len(pert_stream_list)):
    phi1_p, phi2_p = get_phi12_from_stream(pert_stream_list[i][:,:3])
    mass = test_mass[i]
    log_mass = np.log10(mass)
    log_mass_array = np.full_like(phi1_p, log_mass) 
    sc = ax.scatter(phi1_p, phi2_p+offset, s=.1, c=log_mass_array, cmap='plasma_r', norm=norm, rasterized=True)
    offset += -1.9

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(sc, cax=cax)
cbar.set_label(r'$\log_{10}(M_{\mathrm{sh}} / M_\odot)$', fontsize=15)

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-11,10)
ax.set_xlim(-75,25)
ax.set_xlabel(r'$\phi_1$ [deg]',fontsize=20)
ax.set_ylabel(r'$\phi_2$ + c [deg]',fontsize=20)
ax.set_aspect('equal')

In [ ]:
mass_subhalos = jnp.zeros(N_subhalos)
delta_rs = jnp.ones(N_subhalos)#*(-0.04)

test_rs = jnp.linspace(.4,-.4,10)

pert_stream_list_rs = []
for i in tqdm.tqdm(range(len(test_mass))):
    mass_subhalos = mass_subhalos.at[subhalo_view].set(2e7)
    delta_rs = delta_rs.at[subhalo_view].set(test_rs[i])
    stream_pert = compute_pert_stream(stream_base, stream_derivs,mass_subhalos,delta_rs)
    pert_stream_list_rs.append(stream_pert)

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(8,8)

offset = 8.8

# Normalize log_mass values
norm = mcolors.Normalize(vmin=min(test_rs), vmax=max(test_rs))

for i in range(len(pert_stream_list_rs)):
    phi1_p, phi2_p = get_phi12_from_stream(pert_stream_list_rs[i][:,:3])
    delta_rs = test_rs[i]
    test_rs_array = np.full_like(phi1_p, delta_rs) 
    sc = ax.scatter(phi1_p, phi2_p+offset, s=.1, c=test_rs_array, cmap='viridis', norm=norm, rasterized=True)
    offset += -1.9

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(sc, cax=cax)
cbar.set_label(r'$\Delta r_s$ [kpc]', fontsize=15)

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-11,10)
ax.set_xlim(-75,25)
ax.set_xlabel(r'$\phi_1$ [deg]',fontsize=20)
ax.set_ylabel(r'$\phi_2$ + c [deg]',fontsize=20)
ax.set_aspect('equal')

# Fast population realizations from the derivative library

The cells above vary one subhalo at a time. Below we generate many stream realizations with many subhalos 

`streamsculptor.stream_realizations.StreamRealizationGenerator` does this. Per realization it:

1. Draws a Poisson number of impact masses from the SHMF (`RateCalculator.sample_masses`).
2. Matches each drawn mass to the perturber whose `r_s_root` is closest.
3. Assigns the drawn mass to that slot with `delta_r = r_s(m) - r_s_root.
4. sums the linear response — two `einsum`s, fully `vmap`/`scan`-able.


In [ ]:
from streamsculptor.stream_realizations import StreamRealizationGenerator

## Build the realization generator from the get_derivs library + the RateCalculator RC.
realization_gen = StreamRealizationGenerator.from_get_derivs(
                        batches,
                        RateCalculator=RC,
                        log10M_min=6.0, log10M_max=8.5,   # SHMF window (sets the Poisson rate via RC)
                        slope=-1.9, M_hm=0.0, normalization=1.0,
                        mass_fac=1.0,
                        r_s_fac=1.0)                       # subhalo radius scaling: <1 compact, >1 diffuse

## method='vmap' (all at once) or 'scan' (memory-effecient). return_params gives the per-realization draws.
## r_s_fac can also be overridden per call, e.g. generate(..., r_s_fac=0.5), without rebuilding.
out = realization_gen.generate(jax.random.PRNGKey(320), n_realizations=22,
                               method="vmap", return_params=True)

print(f"streams: {out['stream'].shape}   mean impacts/realization: {float(out['n_impact'].mean()):.1f}")

In [ ]:
## Plot a handful of independent CDM realizations, offset in phi2.
n_show = 6
streams = jax.block_until_ready(
    realization_gen.generate(jax.random.PRNGKey(3), n_show, method="vmap")) #9

fig, ax = plt.subplots(1, 1)
fig.set_size_inches(8, 8)

phi1_base, phi2_base = get_phi12_from_stream(stream_base)

ax.scatter(phi1_base, phi2_base + 3, s=.2, color='tab:blue', rasterized=True)
ax.text(22, 3, 'unpert', color='tab:blue', fontsize=12, va='center')
for i in range(n_show):
    phi1_p, phi2_p = get_phi12_from_stream(streams[i][:, :3])
    ax.scatter(phi1_p, phi2_p - i * 3, s=.2, rasterized=True)

ax.tick_params(axis='both', which='major', labelsize=14., length=8)
ax.tick_params(axis='both', which='minor', length=3)
ax.set_ylim(-20, 5)
ax.set_xlim(-75, 20)
ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=20)
ax.set_ylabel(r'$\phi_2$ + c [deg]', fontsize=20)
ax.set_aspect('equal')

# How does GD-1 compare? A lightweight inference with simple summary stats

We now use the fast realization library to ask a population-level question: **are GD-1's
small-scale features consistent with CDM?

We reduce each stream to two summary statistics over a fixed $\phi_1$ window $[-60, 0]$ deg, both
measured **relative to the unperturbed model**:

1. **RMS density fluctuation** — $\Delta \rho(\phi_1)/\rho_0(\phi_1)$
2. **RMS $\phi_2$ track fluctuation** — Compare the model's track to the unperturbed stream track.

(1) requires a KDE. The bandwidth is set once using cross-validation. We also downsample to the number of stars in the data.


In [ ]:
## KDE density helper + binned phi2 track + the reference-based statistic.
## Explicit Gaussian kernel so `bw` IS the absolute bandwidth [deg] -- identical smoothing for the
## data, the model, and every bootstrap draw (gaussian_kde's scalar bw_method is only a factor on
## each dataset's own std, so an absolute bandwidth would need a per-dataset rescale).
_SQRT2PI = jnp.sqrt(2 * jnp.pi)

@jax.jit
def kde_density(phi1, grid, bw):
    """Gaussian-KDE density of phi1 on grid, absolute bandwidth bw [deg]."""
    z = (grid[:, None] - phi1[None, :]) / bw
    return jnp.mean(jnp.exp(-0.5 * z ** 2), axis=1) / (bw * _SQRT2PI)

@jax.jit
def binned_track(phi1, phi2, edges):
    """Mean phi2 per phi1 bin (0 where empty), and the per-bin counts."""
    counts, _ = jnp.histogram(phi1, bins=edges)
    sums, _   = jnp.histogram(phi1, bins=edges, weights=phi2)
    track = jnp.where(counts > 0, sums / jnp.maximum(counts, 1.0), 0.0)
    return track, counts

@jax.jit
def stream_stats(phi1, phi2):
    """(rms_density, rms_track) for a stream, relative to the unperturbed model.
    Density: KDE ratio to RHO0 at bandwidth BW. Track: binned mean-phi2 minus TRACK0_BIN, RMS over
    well-populated body bins. Uses globals GRID, BW, RHO0, W0, EDGES, TRACK0_BIN, BODY_BINS, MIN_COUNT."""
    rho = kde_density(phi1, GRID, BW)
    rms_density = jnp.sqrt(jnp.sum(W0 * (rho / RHO0 - 1.0) ** 2))

    track, counts = binned_track(phi1, phi2, EDGES)
    diff = track - TRACK0_BIN
    valid = BODY_BINS & (counts >= MIN_COUNT)
    rms_track = jnp.sqrt(jnp.sum(jnp.where(valid, diff ** 2, 0.0)) / jnp.maximum(jnp.sum(valid), 1))
    return rms_density, rms_track

In [ ]:
## Fixed window, KDE grid, the in-window GD-1 sample, populations, and the bootstrap driver.
PHI1_MIN, PHI1_MAX = -60.0, 0.0
NGRID = 200
NBINS = 20                                          # phi1 bins for the track statistic


GRID  = jnp.linspace(PHI1_MIN, PHI1_MAX, NGRID)
EDGES = jnp.linspace(PHI1_MIN, PHI1_MAX, NBINS + 1)
BIN_CENTERS = 0.5 * (EDGES[:-1] + EDGES[1:])
MIN_COUNT = 2                                        # min stars for a bin to enter the track RMS

## Unperturbed model track (full resolution) and the in-window GD-1 sample.
phi1_base, phi2_base = get_phi12_from_stream(stream_base)
in_win_data = (np.asarray(phi1_data) >= PHI1_MIN) & (np.asarray(phi1_data) <= PHI1_MAX)
phi1_data_w = np.asarray(phi1_data)[in_win_data]
phi2_data_w = np.asarray(phi2_data)[in_win_data]
N_data_win  = int(in_win_data.sum())               # star count we match the model to
print(f'window [{PHI1_MIN:.0f}, {PHI1_MAX:.0f}] deg;  {N_data_win} GD-1 members in-window')


## Find the optimal KDE bandwidth

In [ ]:
## Choose the density-KDE bandwidth by leave-N_out-out CV ON THE UNPERTURBED MODEL, downsampled
## to GD-1's star count. Then build the references at that bandwidth from the full-res model:
## RHO0 (KDE density) and TRACK0_BIN (binned mean phi2).
N_OUT = 20                                              # stars held out per fold
rng_cv = np.random.default_rng(7)

sel_b  = (np.asarray(phi1_base) >= PHI1_MIN) & (np.asarray(phi1_base) <= PHI1_MAX)
base_w = np.asarray(phi1_base)[sel_b]
cv_samp = base_w[rng_cv.choice(base_w.size, N_data_win, replace=False)]   # model at data's N
n = cv_samp.size
perm  = rng_cv.permutation(n)
folds = [perm[k:k + N_OUT] for k in range(0, n - N_OUT + 1, N_OUT)]       # each holds out N_OUT

dmat = cv_samp[:, None] - cv_samp[None, :]
bw_grid = np.linspace(0.2, 5.0, 50)
cv_ll = np.empty_like(bw_grid)
for j, h in enumerate(bw_grid):
    K = np.exp(-0.5 * (dmat / h) ** 2) / (h * np.sqrt(2 * np.pi))
    ll = 0.0
    for val in folds:                                   # held-out log-likelihood
        train = np.ones(n, bool); train[val] = False
        dens = K[np.ix_(val, np.where(train)[0])].sum(1) / train.sum()
        ll += np.sum(np.log(dens + 1e-300))
    cv_ll[j] = ll / len(folds)
BW = float(bw_grid[np.argmax(cv_ll)])
print(f'leave-{N_OUT}-out CV over {len(folds)} folds -> KDE bandwidth {BW:.2f} deg')

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(bw_grid, cv_ll, color='k', lw=1.8)
ax.axvline(BW, color='tab:red', lw=2, label=f'optimum = {BW:.2f} deg')
ax.set_xlabel('KDE bandwidth [deg]', fontsize=13)
ax.set_ylabel(f'held-out log-likelihood (leave-{N_OUT}-out)', fontsize=12)
ax.set_title('Bandwidth selection on the unperturbed model', fontsize=13)
ax.legend(fontsize=11)
fig.tight_layout(); plt.show()

## Unperturbed-model references (full-resolution base stream, in-window).
phi1_base_w = jnp.asarray(base_w)
phi2_base_w = jnp.asarray(np.asarray(phi2_base)[sel_b])
RHO0 = kde_density(phi1_base_w, GRID, BW)                # density reference
W0   = jnp.where(RHO0 > 0.15 * jnp.max(RHO0), RHO0, 0.0); W0 = W0 / jnp.sum(W0)
TRACK0_BIN, base_counts_bin = binned_track(phi1_base_w, phi2_base_w, EDGES)   # track reference
BODY_BINS = base_counts_bin > 0.15 * jnp.max(base_counts_bin)                 # stream-body bins

## Now setup the realization generator.
We will specify what models we want to generate, and ultimately the number of realizations 

In [ ]:

## Three populations sharing the derivative library.
gen_cdm     = realization_gen = StreamRealizationGenerator.from_get_derivs(
                        batches,
                        RateCalculator=RC,
                        log10M_min=6.0, log10M_max=8.5,   
                        slope=-1.9, M_hm=0.0, normalization=1.0,
                        mass_fac=1.0,
                        r_s_fac=1.0)                    

gen_2x      = StreamRealizationGenerator.from_get_derivs(
                  batches, RateCalculator=RC, log10M_min=6.0, log10M_max=8.5,
                  slope=-1.9, M_hm=0.0, normalization=2.0, mass_fac=1.0, r_s_fac=1.0)

gen_compact = StreamRealizationGenerator.from_get_derivs(
                  batches, RateCalculator=RC, log10M_min=6.0, log10M_max=8.5,
                  slope=-1.9, M_hm=0.0, normalization=1.0, mass_fac=1.0, r_s_fac=0.4)

phi12_batch = jax.jit(jax.vmap(get_phi12_from_stream))   # (R, N, 6) -> (R, N), (R, N)

def population_stats(gen, key, R, D):
    """
    gen: realization generator
    key: jax.random.PRNGKey for subhalo sampling / downsampling
    R: number of stream realizations for the given model
    D: number of bootstrap downsamplings to N_data_win stars each

    returns
    ------
    rms: (R*D,) array of density RMS values
    trk: (R*D,) array of track RMS values
    """
    streams = gen.generate(key, R, method="vmap")
    phi1s, phi2s = phi12_batch(streams)
    phi1s, phi2s = np.asarray(phi1s), np.asarray(phi2s)
    statv = jax.jit(jax.vmap(stream_stats))              # vmap over the D downsamplings
    rng = np.random.default_rng(0)
    rms = np.empty((R, D))
    trk = np.empty((R, D))
   
    for i, (p1_all, p2_all) in enumerate(zip(phi1s, phi2s)):
        # Filter stars in window
        in_win = (p1_all >= PHI1_MIN) & (p1_all <= PHI1_MAX)
        p1, p2 = p1_all[in_win], p2_all[in_win]

        # Draw all (D, N_data_win) bootstrap indices at once
        idx = rng.choice(len(p1), size=(D, N_data_win), replace=True)

        # Compute and store stats
        rms[i], trk[i] = statv(jnp.asarray(p1[idx]), jnp.asarray(p2[idx]))
    return rms.flatten(), trk.flatten()


In [ ]:
# Compute KDE & Track for plotting
rho_d = np.asarray(kde_density(jnp.asarray(phi1_data_w), GRID, BW))
tr_d, cnt_d = binned_track(jnp.asarray(phi1_data_w), jnp.asarray(phi2_data_w), EDGES)
valid = np.asarray(BODY_BINS) & (np.asarray(cnt_d) >= MIN_COUNT)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)


ax0.plot(GRID, rho_d, color='k', lw=1.6, label=f'GD-1 KDE (bw {BW:.2f} deg)')
ax0.plot(GRID, RHO0, color='tab:red', lw=2.2, label=r'unperturbed model $\rho_0$')
ax0.set_ylabel(r'$\rho(\phi_1)$', fontsize=13)
ax0.legend(fontsize=9, loc='upper left')

ax1.scatter(phi1_data_w, phi2_data_w, s=6, color='0.6', alpha=0.6, label='GD-1 members', zorder=-5)
ax1.step(BIN_CENTERS, np.where(valid, np.asarray(tr_d), np.nan), where='mid', color='k', lw=1.6, label=r'GD-1 binned $\phi_2$')
ax1.step(BIN_CENTERS, TRACK0_BIN, where='mid', color='tab:red', lw=2.2, label='unperturbed model track')
ax1.set_ylabel(r'$\phi_2$ [deg]', fontsize=13)
ax1.set_ylim(-4, 4)
ax1.legend(fontsize=9, loc='upper left')

for ax in (ax0, ax1):
    ax.set_xlabel(r'$\phi_1$ [deg]', fontsize=13)
    ax.set_xlim(PHI1_MIN, PHI1_MAX)

fig.tight_layout()


In [ ]:
rms_data, trk_data =  stream_stats(phi1_data_w, phi2_data_w)
rms_data, trk_data

## Run 100 (R) realizations with 50 (D) downsamplings

In [ ]:
## Build the null distributions: R realizations x D bootstrap downsamplings.
R, D = 100, 50 
rms_cdm,  trk_cdm  = population_stats(gen_cdm,     jax.random.PRNGKey(1), R, D)
rms_2x,   trk_2x   = population_stats(gen_2x,      jax.random.PRNGKey(2), R, D)
rms_comp, trk_comp = population_stats(gen_compact, jax.random.PRNGKey(3), R, D)


In [ ]:
## Overlay GD-1 on each null distribution (histograms pool all realizations x downsamplings).
pops = [('CDM', rms_cdm, trk_cdm, 'tab:blue'),
        ('2x CDM', rms_2x, trk_2x, 'tab:orange'),
        ('0.5 rs CDM', rms_comp, trk_comp, 'tab:green')]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, dval, title in [
        (axes[0], 1, rms_data, 'RMS density fluctuation'),
        (axes[1], 2, trk_data, r'RMS $\Delta\phi_2$  [deg]')]:
    lo = min(p[col].min() for p in pops)
    hi = max(max(p[col].max() for p in pops), dval)
    bins = np.linspace(lo, hi * 1.05, 25)
    for name, rms, trk, c in pops:
        s = rms if col == 1 else trk
        ax.hist(s, bins=bins, histtype='step', lw=2, density=True, color=c,
                label=f'{name}')
    ax.axvline(dval, color='k', lw=2.5, label=f'GD-1 data')
    ax.set_xlabel(title, fontsize=13); ax.set_ylabel('density', fontsize=13)
    ax.legend(fontsize=9, loc='upper right')
axes[0].set_title('Density wiggles', fontsize=13)
axes[1].set_title('Track wiggles', fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
from scipy.stats import gaussian_kde

pops = [
    ('CDM', rms_cdm, trk_cdm, 'tab:blue'),
    ('2x CDM', rms_2x, trk_2x, 'tab:orange'),
    ('0.5 rs CDM', rms_comp, trk_comp, 'tab:green'),
]

fig, ax = plt.subplots(figsize=(7, 6))

# Define common evaluation grid for smooth contours
x_all = np.concatenate([rms_cdm, rms_2x, rms_comp, [rms_data]])
y_all = np.concatenate([trk_cdm, trk_2x, trk_comp, [trk_data]])
gx, gy = np.mgrid[
    x_all.min() * 0.9:x_all.max() * 1.1:100j,
    y_all.min() * 0.9:y_all.max() * 1.1:100j
]

# Plot 2D contours for each model
for name, x, y, color in pops:
    kde = gaussian_kde([x, y],bw_method=.5)
    z = kde(np.vstack([gx.ravel(), gy.ravel()])).reshape(gx.shape)
    
    # Draw closed contour levels (e.g., 68% and 95% volume thresholds)
    z_sorted = np.sort(z.ravel())[::-1]
    cumsum = np.cumsum(z_sorted) / z_sorted.sum()
    levels = [z_sorted[np.searchsorted(cumsum, p)] for p in [0.95, 0.68]]
    
    ax.contour(gx, gy, z, levels=sorted(levels), colors=[color], linewidths=[1.2, 2.2])
    # Dummy plot to populate the legend with the model color
    ax.plot([], [], color=color, lw=2, label=name)

# Overlay GD-1 observation
ax.scatter(rms_data, trk_data, color='k', s=80, zorder=5, marker='*', label=f'GD-1')

ax.set_xlabel('RMS Density Fluctuation (Gaps)', fontsize=12)
ax.set_ylabel(r'RMS $\Delta\phi_2$ [deg] (Track Wiggles)', fontsize=12)
ax.set_title('2D Track vs. Density Perturbations', fontsize=13)
ax.legend(frameon=True)
ax.grid(True, alpha=0.3, ls=':')

ax.set_ylim(y_all.min() * 0.9, y_all.max() * 1.1)

plt.tight_layout()
plt.show()

## Experimental: What else can you do with differentiable dynamics x streams?

## Fitting the GD-1 with impulsive subhalo kicks + gradient-based optimization

Basic idea:
- We will generate a stream with 3 subhalo velocity kicks at different times (Plummer spheres)
- The geometry of the impact, mass, radius, and fly-by speed are free parameters for each impact
- We will use gradient-based optimization to try and match the morphology of the GD-1 stream

Essentially, 
$$
\mathrm{model\ stream}
=
\sum_{t_{\rm impact}}
F\!\left(x;\,t_{\rm impact},\theta\right).
$$
We compute a summary statistic: $S(\mathrm{model \ stream})$ and compare it to $S(\mathrm{data \ stream})$. We want to minimize
$$\chi^2 = \left[S(\mathrm{data \ stream})-S(\mathrm{model \ stream} | \theta)\right]^2$$

-------
### Dynamics primer
A subhalo flying past the stream delivers a velocity kick. In the impulse approximation (the star barely moves during the encounter), integrating a Plummer subhalo's acceleration gives a closed form:

$$\Delta\mathbf v = \frac{2GM}{w}\,\frac{\mathbf r_\perp}{|\mathbf r_\perp|^2 + r_s^2},\qquad \mathbf r_\perp = (\mathbf r-\mathbf x_\mathrm{sub}) - \big[(\mathbf r-\mathbf x_\mathrm{sub})\cdot\hat{\mathbf w}\big]\,\hat{\mathbf w},$$

with $M$ the subhalo mass, $w=|\mathbf w|$ its speed relative to the stream, $r_s$ its Plummer scale radius, and $\mathbf r_\perp$ each star's separation from the subhalo's line of flight, projected perpendicular to the direction of motion $\hat{\mathbf w}$. 

In the impulse approximation the mass and fly-by speed are degenrate. We therefore fit the kick amplitude $A \equiv  \frac{2GM}{w}$. The remaining parameters are the scale-radius, $r_s$, and three parameters defining the fly-by geometry (i.e., where in space the closest approach occurs). 

### The kick geometry

We build an orthonormal frame $(\hat T,\hat N,\hat B)$ (via `ssc.impact_frame`) at an anchor point $\mathbf{x_0}$ on the stream track. $\mathbf{x_0}$ is the mean stream position at a chosen $\phi_1$, integrated back to the impact time. $\hat T$ is the along-stream tangent; $\hat N,\hat B$ span the cross-stream plane. Note that $\mathbf x_0$ is the reference for the fit parameters, it is not itself the closest-approach point. The five per-kick numbers $[A, s_0, b_N, b_B, r_s]$ place the fly-by relative to $\mathbf x_0$. Parameters $s_0, b_N, b_B$ define the location of the impact along the stream.



In [ ]:
from streamsculptor.streamhelpers import gen_stream_ics_Chen25

# Set random seed and solver parameters for stream generation
KEY   = jax.random.PRNGKey(0)
SOLVE = dict(solver=diffrax.Dopri5(), rtol=1e-6, atol=1e-6, dtmin=0.5)

# unperturbed stream: leading + trailing particle at each stripping time, packed [x, v]
(pos_lead, pos_trail, vel_lead, vel_trail), _ = gen_stream_ics_Chen25(
    pot_base=pot, ts=ts, prog_w0=IC, Msat=1e4, key=KEY, **SOLVE)

# initial conditions of UNPERTURBED stream
# -1 because the last particle is the progenitor 
Y0_UNPERT = jnp.concatenate([jnp.hstack([pos_lead[:-1],  vel_lead[:-1]]),
                             jnp.hstack([pos_trail[:-1], vel_trail[:-1]])], axis=0)
T0_UNPERT = jnp.concatenate([ts[:-1], ts[:-1]])          # release time per particle

def plummer_kick_stream(params, t_imps, xi0s, Yin=None, Tin=None):
    """
    Sum of Plummer kicks 
    params (N,5)=[A,s0,b_N,b_B,r_s], anchored at phi1=xi0s, applied at t_imps.
    """
    if Yin is None: 
        Yin, Tin = Y0_UNPERT, T0_UNPERT
    # where the anchor is in phi1 for each impact
    w_anchors = jax.vmap(eval_mean)(xi0s)
    return ssc.perp_kick_stream_curved(pot, Yin, Tin, ts[-1], params, t_imps, w_anchors, **SOLVE)

# downsample the stream (to 900 particles) for cheaper gradient steps.
# We will upsample later to visualize the fit 
downsample_idx = jnp.linspace(0, Y0_UNPERT.shape[0]-1, 900).astype(int)
Y0_FIT, T0_FIT = Y0_UNPERT[downsample_idx], T0_UNPERT[downsample_idx]          # subset for cheap gradient steps

Pick a few impact times and crossing points, and eyeball an initial kick amplitude/geometry.

In [ ]:
N_PLUM = 3
T_IMPS =  jnp.array([-400., -300., -200.])          # impact times [Myr]
XI0S   = jnp.array([-35.,  -30.,  -25.])           # crossing phi1 [deg]

p_eye  = jnp.array([[0.004, 0.0, 0.06, 0.02, 0.12]] * N_PLUM)   # [A, s0, b_N, b_B, r_s]



stream_eye = plummer_kick_stream(p_eye, T_IMPS, XI0S)
phi1_eye, phi2_eye = get_phi12_from_stream(stream_eye)


fig, ax = plt.subplots(figsize=(11, 3.2))
ax.scatter(phi1_data, phi2_data, s=5, color="k", label="data")
ax.scatter(phi1_eye, phi2_eye, s=1.5, color="tab:green", alpha=0.4, label="by-eye kicks", rasterized=True)
ax.set_xlim(-55, -20); ax.set_ylim(-3, 3); ax.legend(markerscale=4)
ax.set_xlabel(r"$\phi_1$ [deg]"); ax.set_ylabel(r"$\phi_2$ [deg]"); plt.show()

In [ ]:
p_eye

We compress data and model into a smooth two-branch summary statistic at 7 nodes along the stream: 
- the track ridge (main stream)
- the spur ridge
- the spur fraction. We use a soft window + sigmoid to determine what belongs to stream/spur to keep it differentiable.

In [ ]:
NODES = jnp.linspace(-40., -20., 7)
NS = 3.0
SPLIT = 0.6 # where we draw the stream/spur dividing line

def branch_summary(phi1, phi2, w):
    def at(n):
        k = jnp.exp(-0.5*((phi1 - n)/NS)**2) * w
        s = jax.nn.sigmoid((phi2 - SPLIT)/0.15)                      # soft spur (1) vs track (0)
        return jnp.array([jnp.sum(k*(1-s)*phi2)/(jnp.sum(k*(1-s)) + 1e-6),   # track ridge
                          jnp.sum(k*s*phi2)    /(jnp.sum(k*s)     + 1e-6),   # spur ridge
                          jnp.sum(k*s)      /(jnp.sum(k)       + 1e-6)])  # spur fraction
    return jax.vmap(at)(NODES)

in_box = ((phi1_data > -42) & (phi1_data < -26)).astype(float)   # 1 for data in the spur window, 0 outside
TARGET = branch_summary(phi1_data, phi2_data, in_box)
weights  = jnp.sqrt(jnp.array([1.0, 1.0, 6.0]))[None, :]            # weight spur fraction 6x

p0 = p_eye.ravel()

@jax.jit
def kick_resid(pvec):
    phi1, phi2 = get_phi12_from_stream(plummer_kick_stream(pvec.reshape(p_eye.shape), T_IMPS, XI0S, Y0_FIT, T0_FIT))
    r = (weights*(branch_summary(phi1, phi2, jnp.ones_like(phi1)) - TARGET)).ravel()
    return jnp.where(jnp.isfinite(r), r, 10.0)
    
print("cost at by-eye: %.3f" % float(jnp.sum(kick_resid(p0)**2)))

In [ ]:
# the data ridges + spur fraction the fit targets
S_data = np.asarray(TARGET); nodes = np.asarray(NODES)

fig, ax = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax[0].scatter(phi1_data, phi2_data, s=5, color="k", alpha=0.5, label="GD-1 data")

ax[0].axhline(SPLIT, ls=":", color="gray", lw=0.8)
ax[0].plot(nodes, S_data[:, 0], "o-", color="tab:blue",   label="track ridge")
ax[0].plot(nodes, S_data[:, 1], "o-", color="tab:orange", label="spur ridge")
ax[0].set_ylim(-3, 3)
ax[0].set_ylabel(r"$\phi_2$ [deg]")

ax[0].legend(ncol=3, fontsize=9, loc="lower left")
ax[1].plot(nodes, S_data[:, 2], "o-", color="k")
ax[1].set_ylabel("spur\nfraction")
ax[1].set_ylim(0, None)
ax[1].set_xlabel(r"$\phi_1$ [deg]")
ax[1].set_xlim(-50, -18)
fig.suptitle("Data summary (fit target)")
plt.show()

In [ ]:
kick_resid(p_eye)

In [ ]:
p_eye.shape

In [ ]:
## we can take a gradient of the summary with respect to model params:
jax.jacfwd(kick_resid)(p_eye)

Fit the 5 params per kick with Levenberg-Marquardt (using the jax package optimistix).

In [ ]:
import optimistix as optx, time
solver = optx.LevenbergMarquardt(rtol=1e-6, atol=1e-6)
t0  = time.time()
sol = optx.least_squares(lambda p, args: kick_resid(p), solver, p0, max_steps=100, throw=False)
kick_fit = sol.value.reshape(p_eye.shape)

## Summarize
print("%.0fs.  cost %.3f -> %.3f" % (time.time()-t0,
      float(jnp.sum(kick_resid(p0)**2)), float(jnp.sum(kick_resid(sol.value)**2))))
for k in range(N_PLUM):
    print("  kick %d @ t=%+.0f: %s" % (k, float(T_IMPS[k]), np.round(np.asarray(kick_fit[k]), 4)))

In [ ]:
phi1_fit, phi2_fit = get_phi12_from_stream(plummer_kick_stream(kick_fit, T_IMPS, XI0S))
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].scatter(phi1_data, phi2_data, s=5, color="k"); axes[0].scatter(phi1_eye, phi2_eye, s=5, color="tab:orange", label="by-eye"); axes[0].set_title("by-eye kicks"); axes[0].legend(markerscale=4)
axes[1].scatter(phi1_data, phi2_data, s=5, color="k"); axes[1].scatter(phi1_fit, phi2_fit, s=3, color="tab:red", label="fit"); axes[1].set_title("fitted kicks"); axes[1].legend(markerscale=4)

for ax in axes: 
    ax.set_xlim(-55, -20)
ax.set_ylim(-3, 3)
ax.set_ylabel(r"$\phi_2$ [deg]")
ax.axhline(0.6, ls=":", color="gray", lw=0.8)
axes[1].set_xlabel(r"$\phi_1$ [deg]")
plt.show()

In [ ]:
# data (solid) vs fit (dashed) summary
S_model = branch_summary(phi1_fit, phi2_fit, jnp.ones_like(phi1_fit))


fig, ax = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax[0].scatter(phi1_data, phi2_data, s=5, color="k", alpha=0.5, label="GD-1 data")

ax[0].axhline(SPLIT, ls=":", color="gray", lw=0.8)

ax[0].plot(nodes, S_data[:, 0], "o-", color="tab:blue",   label="track (data)")
ax[0].plot(nodes, S_data[:, 1], "o-", color="tab:orange", label="spur (data)")
ax[0].plot(nodes, S_model[:, 0], "s--", color="tab:blue",   mfc="none", label="track (fit)")
ax[0].plot(nodes, S_model[:, 1], "s--", color="tab:orange", mfc="none", label="spur (fit)")

ax[0].set_ylim(-3, 3)
ax[0].set_ylabel(r"$\phi_2$ [deg]")
ax[0].legend(ncol=2, fontsize=8, loc="lower left")
ax[1].plot(nodes, S_data[:, 2],  "o-",  color="k",       label="data")
ax[1].plot(nodes, S_model[:, 2], "s--", color="tab:red", mfc="none", label="fit")
ax[1].set_ylabel("spur\nfraction")

ax[1].set_ylim(0, None)
ax[1].legend(fontsize=8)
ax[1].set_xlabel(r"$\phi_1$ [deg]")
ax[1].set_xlim(-42, -18)


Convert each fitted amplitude to a subhalo mass via $A = 2GM/w$. Only $M/w$ is constrained, so pick a crossing speed $w$.

In [ ]:
from astropy import constants as const
G = (const.G).to('kpc3 / (Msun Myr2)').value
w_assumed = (150*u.km/u.s).to('kpc/Myr').value     # M scales linearly with w
print("implied subhalos (w = 150 km/s):")
print("  %-4s %-8s %-11s %-8s %-8s" % ("kick", "t[Myr]", "M[Msun]", "b[kpc]", "r_s[kpc]"))
for k in range(N_PLUM):
    A, s0, bN, bB, rs = np.asarray(kick_fit[k])
    print("  %-4d %-8.0f %-11.2e %-8.3f %-10.9f" % (k, float(T_IMPS[k]), abs(A*w_assumed/(2*G)), np.hypot(bN,bB), abs(rs)))

# Adding in time-dependence in the potential

Below we will integrate orbits in a custom time-dependent potential function. 
All we need to supply is the potential: Jax will take care of its derivatives via automatic differentiation.


The potential function we will define is a time-dependent NFW

$$\Phi_{\rm osc}\left(\mathbf{r}, t\right) = -\frac{G M(t)}{r^\prime} \ln{\left(1 + \frac{r^\prime}{r_s(t)}\right)}$$
with the functions
$$M(t) = M_{\rm base} + M_{\rm osc}\sin\left(\Omega t\right)$$
$$r_s(t) = r_{s,\rm base} + r_{s, \rm osc}\sin\left(\Omega t\right)$$
$$r^\prime = \sqrt{x^2 + y^2 + \left(\frac{z}{q(t)}\right)^2}$$
$$q(t) = 1 + 0.3 \times \cos{\left(\Omega t\right)}$$




In [ ]:
#from astropy.constants import G
## Define the grav. constant in the simulation unit system [kpc, Myr, Msun]
G_ = usys.G#G.decompose(usys).value
M_base = 1e12 #Msun
M_osc = 5e11
rs_base = 15.0 #kpc
rs_osc = 10.0
T = 500.0 #Myr
omega = 2*jnp.pi / T

## Define the oscilatting potential function
@jax.jit
def osc_potential_func(xyz, t):
    xyz = jnp.array([xyz[0], xyz[1], xyz[2]/(1 + 0.3*jnp.cos(omega*t))])
    r = jnp.sqrt(jnp.sum(xyz**2))

    curr_mass = M_base + M_osc*jnp.sin(omega*t)
    rs_curr = rs_base + rs_osc*jnp.sin(omega*t)
    return - (G_*curr_mass / r)*jnp.log(1 + r/rs_curr)
    
## Define its static counterpart
@jax.jit
def static_potential_func(xyz, t):
    r = jnp.sqrt(jnp.sum(xyz**2))
    curr_mass = M_base 
    rs_curr = rs_base 
    return - (G_*curr_mass / r)*jnp.log(1 + r/rs_curr)

In [ ]:
from streamsculptor.potential import CustomPotential
## Create the CustomPotential objects. All we need to do is tell 
## the CustomPotential object what the potential function is. Jax will take care of the rest.
pot_osc = CustomPotential(potential_func=osc_potential_func, units=usys)
pot_static = CustomPotential(potential_func=static_potential_func, units=usys)

In [ ]:
## Compute potential at different times –– should be different:
xyz = jnp.array([1.0, 20.0, 10.0])
print(pot_osc.potential(xyz,-1000.0),pot_osc.potential(xyz,-850.0)) 


In [ ]:
## Let's visualize the density of both potentials, by taking the Laplacian of the potential using autodiff
x_grid = jnp.linspace(-15, 15, 100)
z_grid = jnp.linspace(-15, 15, 100)
X, Z = jnp.meshgrid(x_grid, z_grid)
inp = jnp.array([X.flatten(), jnp.zeros_like(X.flatten())+5.0, Z.flatten()]).T 

## Time dep case:
t1, t2, t3 = -1500.0, -1200, -1050.0
## Most basic functions in StreamSculptor assume a 1D input.
## But we can still evalaute over many inputs using the vmap (vectorized map) functionality from Jax
## The syntax is batch_function = jax.vmap(function, in_axes=(tuple of axes to map over))
dens_t1 = jax.vmap(pot_osc.density,in_axes=(0,None))(inp, t1).reshape(X.shape)
dens_t2 = jax.vmap(pot_osc.density,in_axes=(0,None))(inp, t2).reshape(X.shape)
dens_t3 = jax.vmap(pot_osc.density,in_axes=(0,None))(inp, t3).reshape(X.shape)

## Static case:
dens_static = jax.vmap(pot_static.density,in_axes=(0,None))(inp, 0.0).reshape(X.shape)

## Visualize
fig, ax = plt.subplots(2,3, figsize=(10,5))

ax[0,0].contourf(X,Z,dens_t1,levels=4,cmap='Blues')
ax[0,1].contourf(X,Z,dens_t2,levels=4,cmap='Blues')
ax[0,2].contourf(X,Z,dens_t3,levels=4,cmap='Blues')

for i in range(3):
    ax[1,i].contourf(X,Z,dens_static,levels=4,cmap='Blues')
    ax[0,i].text(0.1,0.85,f'{[t1,t2,t3][i]}' + " Myr",transform=ax[0,i].transAxes,fontsize=18)
    ax[0,i].text(0.1,0.1, r"$\rho_{\rm osc}$",transform=ax[0,i].transAxes,fontsize=18)
    ax[1,i].text(0.1,0.1, r"$\rho_{\rm static}$",transform=ax[1,i].transAxes,fontsize=18)
ax_flat = ax.flatten()

for i in range(6):
    ax_flat[i].set_aspect('equal')  

fig.subplots_adjust(wspace=-0.3,hspace=0.2)



In [ ]:
## now integrate some orbits
w0 = jnp.array([10.0, 10.0, 15.0, 0.1, 0.2, 0.1])
orb_osc = pot_osc.integrate_orbit(w0=w0,ts=ts).ys
orb_static = pot_static.integrate_orbit(w0=w0,ts=ts).ys

In [ ]:
fig, ax = plt.subplots(1,1)
fig.set_size_inches(6,6)
ax.plot(orb_static[:,1],orb_static[:,2], label='Static Potential',lw=2)
ax.plot(orb_osc[:,1],orb_osc[:,2],label=r'Oscillating Potential: $M(t), r_s(t)$',lw=2)

ax.legend(fontsize=17,loc='lower left',frameon=True)
ax.set_xlabel('y [kpc]', fontsize=20)
ax.set_ylabel('z [kpc]', fontsize=20)

ax.set_aspect('equal')


In [ ]:
jnp.array([ts.min()])

In [ ]:
## Now generate a stream in both potentials
## Get initial conditions, such that the final location of the progenitor is the same (at t = 0)
IC_osc = pot_osc.integrate_orbit(w0=prog_wtoday,t0=0, t1=ts.min(), ts=jnp.array([ts.min()])).ys[0]
IC_static = pot_static.integrate_orbit(w0=prog_wtoday,t0=0, t1=ts.min(), ts=jnp.array([ts.min()])).ys[0]

## Generate streams in oscillating and static potentials
l, t = ssc.gen_stream_Chen25(pot_base=pot_osc, ts=ts, prog_w0=IC_osc, Msat=1e4,
                             key=jax.random.PRNGKey(532), method="vmap",
                             solver=diffrax.Dopri8(), atol=1e-7, rtol=1e-7, dtmin=0.1)
stream_osc = jnp.vstack([l, t])


l, t = ssc.gen_stream_Chen25(pot_base=pot_static, ts=ts, prog_w0=IC_static, Msat=1e4,
                             key=jax.random.PRNGKey(532), method="vmap",
                             solver=diffrax.Dopri8(), atol=1e-7, rtol=1e-7, dtmin=0.1)
stream_static = jnp.vstack([l, t])


In [ ]:
phi1_static, phi2_static = get_phi12_from_stream(stream_static)
phi1_osc, phi2_osc = get_phi12_from_stream(stream_osc)


fig, ax = plt.subplots(figsize=(8, 3.2))
ax.scatter(phi1_static, phi2_static, s=5, color="k", label="static")
ax.scatter(phi1_osc, phi2_osc-4, s=1.5, color="tab:green", alpha=0.4, label="by-eye kicks", rasterized=True)
ax.set_xlim(-80, 15)

ax.set_ylim(-11, 3)
#ax.legend(markerscale=4)
ax.set_aspect('equal')
ax.set_xlabel(r"$\phi_1$ [deg]")
ax.set_ylabel(r"$\phi_2$ [deg]"); plt.show()